# CNN Image Classification — CIFAR-10

Dataset used: **CIFAR-10** (10 classes, 32×32 RGB images, 50,000 train / 10,000 test).
*Assumption note: the task sheet does not name a dataset — CIFAR-10 is used here since it matches the 10-class, 32×32 image description. If a different dataset was assigned (e.g. Fashion-MNIST, CIFAR-100),*


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)


## Task 1: Understand the Dataset

In [ ]:
# Load the dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Number of classes:", len(class_names))
print("Image size:", x_train.shape[1:])
print("Number of training images:", x_train.shape[0])
print("Number of testing images:", x_test.shape[0])
print("Class names:", class_names)


In [ ]:
# Display the first 10 images with labels
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[int(y_train[i])])
    plt.axis('off')
plt.suptitle("First 10 Training Images")
plt.tight_layout()
plt.show()


## Task 2: Data Preprocessing 

In [ ]:
# Normalize pixel values (0-255 -> 0-1)
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm = x_test.astype('float32') / 255.0

# One-hot encode labels
num_classes = 10
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

# Verify shapes
print("X_train shape:", x_train_norm.shape)
print("X_test shape:", x_test_norm.shape)
print("y_train shape:", y_train_cat.shape)
print("y_test shape:", y_test_cat.shape)


## Task 3: Build a CNN Model

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')  # Output layer
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


## Task 4: Train the Model 

In [ ]:
history = model.fit(
    x_train_norm, y_train_cat,
    batch_size=64,
    epochs=25,
    validation_data=(x_test_norm, y_test_cat)
)


## Task 5: Evaluate the Model 

In [ ]:
train_loss, train_acc = model.evaluate(x_train_norm, y_train_cat, verbose=0)
test_loss, test_acc = model.evaluate(x_test_norm, y_test_cat, verbose=0)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy:  {test_acc:.4f}")
print(f"Training Loss:     {train_loss:.4f}")
print(f"Testing Loss:      {test_loss:.4f}")

gap = train_acc - test_acc
if gap > 0.10:
    print("\nComment: Training accuracy is notably higher than testing accuracy "
          "-> the model is OVERFITTING (it has memorized the training data and "
          "generalizes less well to unseen data).")
elif train_acc < 0.6 and test_acc < 0.6:
    print("\nComment: Both training and testing accuracy are low -> the model is "
          "UNDERFITTING (it hasn't learned the patterns in the data well enough; "
          "consider a deeper network, more epochs, or a lower learning rate).")
else:
    print("\nComment: Training and testing accuracy are close and reasonably high "
          "-> the model is fitting well, with only mild/no overfitting.")


## Task 6: Plot Performance Graphs 

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


**Explanation of the graphs:**
- The **accuracy plot** shows how correctly the model classifies images over training epochs, for both the training set and the held-out validation (test) set. If the training curve keeps rising while the validation curve flattens or drops, that's a sign of overfitting.
- The **loss plot** shows how the categorical crossentropy loss changes. Ideally both training and validation loss decrease and converge; if validation loss starts rising while training loss keeps falling, the model is overfitting past that epoch.


## Task 7: Predict New Images 

In [ ]:
# Select 10 test images
indices = np.random.choice(len(x_test_norm), 10, replace=False)
sample_images = x_test_norm[indices]
sample_labels = y_test[indices].flatten()

predictions = model.predict(sample_images)
predicted_labels = np.argmax(predictions, axis=1)

plt.figure(figsize=(15, 6))
for i, idx in enumerate(indices):
    plt.subplot(2, 5, i + 1)
    plt.imshow(sample_images[i])
    actual = class_names[sample_labels[i]]
    predicted = class_names[predicted_labels[i]]
    correct = actual == predicted
    color = 'green' if correct else 'red'
    plt.title(f"Actual: {actual}\nPred: {predicted}", color=color, fontsize=9)
    plt.axis('off')
plt.suptitle("Predictions on 10 Test Images (red = incorrect)")
plt.tight_layout()
plt.show()


## Task 8: Improve the CNN 
Three improvements applied here:
1. **Extra Conv2D layer** (deeper feature extraction)
2. **Batch Normalization** (stabilizes/speeds up training)
3. **Dropout** (reduces overfitting)


In [ ]:
improved_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),  # extra Conv2D layer
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),  # Dropout
    layers.Dense(num_classes, activation='softmax')
])

improved_model.compile(optimizer='adam',
                        loss='categorical_crossentropy',
                        metrics=['accuracy'])

improved_model.summary()

improved_history = improved_model.fit(
    x_train_norm, y_train_cat,
    batch_size=64,
    epochs=25,
    validation_data=(x_test_norm, y_test_cat)
)


In [ ]:
_, old_test_acc = model.evaluate(x_test_norm, y_test_cat, verbose=0)
_, new_test_acc = improved_model.evaluate(x_test_norm, y_test_cat, verbose=0)

print(f"Old Model Test Accuracy: {old_test_acc:.4f}")
print(f"New Model Test Accuracy: {new_test_acc:.4f}")
print(f"Improvement: {(new_test_acc - old_test_acc) * 100:.2f} percentage points")


## Task 9: Confusion Matrix

In [ ]:
y_pred = improved_model.predict(x_test_norm)
y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = y_test.flatten()

cm = confusion_matrix(y_true_labels, y_pred_labels)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix')
plt.show()

print(classification_report(y_true_labels, y_pred_labels, target_names=class_names))


**Precision, Recall, F1-score:**
- **Precision** = TP / (TP + FP) — of all images the model predicted as a given class, the fraction that were actually that class. High precision means few false alarms.
- **Recall** = TP / (TP + FN) — of all images that truly belong to a class, the fraction the model correctly identified. High recall means few misses.
- **F1-score** = harmonic mean of precision and recall — a single balanced measure, useful when you care about both false positives and false negatives, especially with imbalanced classes.


## Task 10: Model Analysis

**1. Why is CNN preferred over ANN for image classification?**
CNNs use convolutional filters that scan across the image, so they capture spatial and local patterns (edges, textures, shapes) while sharing weights across the image. A plain ANN would need to flatten the image into a 1D vector immediately, losing spatial structure and requiring far more parameters, making it both less accurate and less efficient for image data.

**2. What is the role of Conv2D?**
The Conv2D layer applies learnable filters (kernels) that slide over the input image to detect local features such as edges, corners, and textures. Each filter produces a feature map highlighting where that pattern appears in the image.

**3. Why is MaxPooling used?**
MaxPooling downsamples feature maps by taking the maximum value in each region, reducing spatial dimensions. This lowers computation, controls overfitting, and gives the network some translation invariance (small shifts in the image don't change the output much).

**4. Why is Flatten required?**
Convolution and pooling layers output multi-dimensional feature maps (height × width × channels). The Flatten layer reshapes this into a single 1D vector so it can be fed into the fully connected Dense layers, which expect 1D input.

**5. What does Softmax do?**
Softmax is used in the output layer to convert the raw output scores (logits) into a probability distribution over the classes — all values are between 0 and 1 and sum to 1, making the largest value interpretable as the model's predicted class.

**6. Which optimizer is used?**
The **Adam** optimizer, which combines momentum and adaptive per-parameter learning rates for fast, stable convergence.

**7. Which loss function is used?**
**Categorical Crossentropy**, appropriate for multi-class classification with one-hot encoded labels.

**8. How can overfitting be reduced?**
- Add Dropout layers
- Use Batch Normalization
- Apply data augmentation (rotations, flips, shifts)
- Use L1/L2 regularization
- Use EarlyStopping to stop training once validation loss stops improving
- Reduce model complexity or increase training data
- Use Cross-validation to tune hyperparameters more robustly
